Our goal in this Jupyter Notebook is to generate a Gaussian copula linking returns, but drawing samples from an alternative (t) distribution. I.e. a t distribution with Gaussian copula.

In [79]:
import pandas as pd
import numpy as np

In [80]:
df = pd.read_excel("Archegos.xlsx", sheet_name = "Data6m")
df["Date"] = pd.to_datetime(df["Date"], format = "%d/%m/%Y")
df = df.set_index("Date")

In [81]:
print(df.head())
print(df.iloc[-1])

              BIDU    TME   VIPS       FTCH     IQ    PARA  DISCA    GSX  \
Date                                                                       
2021-03-22  266.13  30.87  45.25  62.000000  28.04  100.34  74.65  83.79   
2021-03-19  257.47  30.42  45.45  59.880001  27.77   97.35  77.27  86.60   
2021-03-18  264.85  28.52  44.46  59.500000  26.63   96.76  75.95  89.68   
2021-03-17  277.13  27.90  44.11  60.770000  26.03   92.34  74.07  92.67   
2021-03-16  266.78  26.45  42.92  59.250000  25.50   96.24  75.81  89.30   

                 NDX      S&P  
Date                           
2021-03-22  13086.51  3940.59  
2021-03-19  12866.99  3913.10  
2021-03-18  12789.14  3915.46  
2021-03-17  13202.38  3974.12  
2021-03-16  13152.28  3962.71  
BIDU      107.10
TME        12.60
VIPS       17.14
FTCH       13.92
IQ         16.70
PARA       21.53
DISCA      21.95
GSX        32.61
NDX      9598.89
S&P      3055.73
Name: 2020-06-01 00:00:00, dtype: float64


In [82]:
correlations = pd.read_pickle("correlations.pkl")
correlations_2 = pd.read_pickle("correlations_2.pkl")
vols = pd.read_pickle("vols.pkl")
means = pd.read_pickle("means.pkl")
# correlations = np.zeros(shape = (10, 10)) + 0.99 + np.identity(10) * 0.01
# correlations = pd.DataFrame(correlations)
print(correlations)

           BIDU       TME      VIPS      FTCH        IQ      PARA     DISCA  \
BIDU   1.000000  0.277230  0.183607  0.233096  0.165643 -0.148865 -0.087102   
TME    0.277230  1.000000  0.179014  0.160251  0.145964 -0.134262 -0.152444   
VIPS   0.183607  0.179014  1.000000  0.142022  0.130518 -0.172356 -0.138265   
FTCH   0.233096  0.160251  0.142022  1.000000 -0.005845 -0.140970 -0.165009   
IQ     0.165643  0.145964  0.130518 -0.005845  1.000000 -0.112268 -0.067519   
PARA  -0.148865 -0.134262 -0.172356 -0.140970 -0.112268  1.000000  0.752806   
DISCA -0.087102 -0.152444 -0.138265 -0.165009 -0.067519  0.752806  1.000000   
GSX    0.085910  0.151511  0.084921  0.053516  0.135034  0.078133  0.046718   
NDX    0.432392  0.272823  0.217171  0.436955  0.171860 -0.042907 -0.027795   
S&P    0.377446  0.196029  0.146626  0.394372  0.173959  0.172132  0.248047   

            GSX       NDX       S&P  
BIDU   0.085910  0.432392  0.377446  
TME    0.151511  0.272823  0.196029  
VIPS   0.084921 

In [83]:
print(correlations_2)

           BIDU       TME      VIPS      FTCH        IQ      PARA     DISCA  \
BIDU   1.000000  0.291189  0.199336  0.250802  0.173106 -0.113877 -0.058945   
TME    0.291189  1.000000  0.198104  0.184016  0.154981 -0.093305 -0.116681   
VIPS   0.199336  0.198104  1.000000  0.166178  0.139713 -0.130125 -0.103143   
FTCH   0.250802  0.184016  0.166178  1.000000  0.007776 -0.091301 -0.121295   
IQ     0.173106  0.154981  0.139713  0.007776  1.000000 -0.091899 -0.051280   
PARA  -0.113877 -0.093305 -0.130125 -0.091301 -0.091899  1.000000  0.764065   
DISCA -0.058945 -0.116681 -0.103143 -0.121295 -0.051280  0.764065  1.000000   
GSX    0.093682  0.159803  0.094104  0.065125  0.139318  0.091714  0.059410   
NDX    0.439147  0.283106  0.228312  0.445265  0.177617 -0.019000 -0.007624   
S&P    0.385945  0.209134  0.160528  0.405337  0.180346  0.191664  0.263362   

            GSX       NDX       S&P  
BIDU   0.093682  0.439147  0.385945  
TME    0.159803  0.283106  0.209134  
VIPS   0.094104 

In [84]:
print(vols)

              0
BIDU   0.568581
TME    0.465502
VIPS   0.507060
FTCH   0.638710
IQ     0.566297
PARA   0.523482
DISCA  0.481606
GSX    1.099691
NDX    0.255234
S&P    0.186645


In [85]:
print(means)

              0
BIDU   1.327444
TME    1.256045
VIPS   1.371928
FTCH   2.117226
IQ     0.824081
PARA   2.108300
DISCA  1.683712
GSX    1.813333
NDX    0.429533
S&P    0.343138


Using these means is probably not valid, since it was as a result of the price impact of the trades made by Archegos that caused such large rises in the first place

In [86]:
from scipy.linalg import cholesky

correlations_chol = cholesky(correlations.values, lower = True)

In [87]:
correlations_reconstructed = correlations_chol @ correlations_chol.T
print(correlations_reconstructed)

[[ 1.          0.27723031  0.18360719  0.23309605  0.1656428  -0.14886536
  -0.08710169  0.08591008  0.43239201  0.37744612]
 [ 0.27723031  1.          0.1790137   0.16025143  0.14596448 -0.13426163
  -0.15244363  0.15151078  0.27282251  0.19602882]
 [ 0.18360719  0.1790137   1.          0.14202221  0.13051844 -0.17235593
  -0.13826542  0.08492079  0.21717092  0.14662601]
 [ 0.23309605  0.16025143  0.14202221  1.         -0.00584484 -0.14097031
  -0.16500944  0.05351575  0.43695486  0.39437218]
 [ 0.1656428   0.14596448  0.13051844 -0.00584484  1.         -0.11226766
  -0.06751945  0.13503373  0.17186001  0.17395875]
 [-0.14886536 -0.13426163 -0.17235593 -0.14097031 -0.11226766  1.
   0.75280643  0.07813265 -0.04290718  0.1721319 ]
 [-0.08710169 -0.15244363 -0.13826542 -0.16500944 -0.06751945  0.75280643
   1.          0.04671755 -0.02779546  0.24804705]
 [ 0.08591008  0.15151078  0.08492079  0.05351575  0.13503373  0.07813265
   0.04671755  1.          0.18075171  0.06151166]
 [ 0.432

In [88]:
def multivariate_samples(samples, correlations):
    correlations_chol = cholesky(correlations.values, lower = True)

    n, m = correlations.shape

    normals = np.random.standard_normal(size = (n, samples))

    return correlations_chol @ normals
    

In [89]:
multivariate_sampler = multivariate_samples(2 , correlations)

print(multivariate_sampler)


[[ 1.71468636  0.39475655]
 [ 1.88777408  0.23788522]
 [-0.36151958 -1.11090039]
 [ 1.45050793 -0.91762113]
 [-0.42070133  0.86446696]
 [-0.80721186 -0.16190732]
 [-1.33272054  0.96233458]
 [-0.05733219  0.38120756]
 [ 1.9272086  -1.23141659]
 [ 1.39961137 -0.90790687]]


In [90]:
stocks = ["BIDU", "TME", "VIPS", "FTCH", "IQ", "PARA", "DISCA", "GSX", "NDX", "S&P"]
mus = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0] # Daily means
sigmas = [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01] # Daily sigmas
degrees_of_freedoms = [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]



In [91]:
from scipy.stats import norm
from scipy.stats import t as t_dist

In [92]:
def t_log_returns(multivariate_sampler, stocks, mus, sigmas, degrees_of_freedoms):
    """
    Function to take a set of multivariate Gaussian samples, and return t-distribution samples
    distributed according to the Gaussian copula implied by the multivariate Gaussian samples.
    
    Multivariate samples are standardised multivariate samples
    Stocks are the names of the stocks sampled
    mus are the daily average drifts for the stocks
    sigmas are the daily volatilities for the stocks
    degrees of freedoms are the degrees of freedom for each stock
    """
    n = len(stocks)

    log_returns = np.zeros(shape = multivariate_sampler.shape)
    
    for i in range(0, n):
        log_returns[i, :] = t_dist.ppf(norm.cdf(multivariate_sampler[i, :]), df = degrees_of_freedoms[i]) * sigmas[i] + mus[i]

    return log_returns

In [93]:
def generate_t_increment_portfolio_changes_paths_frame(stocks, mus, sigmas, degrees_of_freedoms, correlations, MPOR, initial_values, samples):
    """
    Given a portfolio of stocks, daily drifts, daily volatilities, degrees of freedom, MPOR, 
    initial values for stock prices, and quantities of shares held returns a samples of 
    portfolio value changes over the MPOR distributing the stocks according to t-distributions
    correlated according to a Gaussian copula with correlation matrix correlations.

    stocks: A list containing strings for the name of each stock
    mus: The daily drifts of each stock A np array of floats
    sigmas: The daily vols of each stock. A np array of floats
    degrees_of_freedoms: The t-distribution number of degrees of freedom for each stock. A fnp array of floats
    correlations: The correlation matrix of the underlying Gaussian copula. A np array or pandas matrix
    MPOR: The margin period of risk. An integer
    initial_values: The starting stock price of each stock. A np array of floats.
    quantities: The quantity of each stock held (long or short) A np array of floats.
    samples: The number of samples to generate
    """
    
    no_of_stocks = len(stocks)

    increments = np.zeros(shape = (samples, no_of_stocks))

    for i in range(0, samples):
        gaussian_samples = multivariate_samples(MPOR, correlations)

        t_log_returns_array = t_log_returns(gaussian_samples, stocks, mus, sigmas, degrees_of_freedoms)

        overall_returns = t_log_returns_array.sum(axis = 1)

        increments[i, :] = (np.exp(overall_returns) - 1) * initial_values
    
    portfolio_increment_frame = pd.DataFrame(increments, columns = stocks)

    portfolio_increment_frame['Total'] = portfolio_increment_frame.sum(axis = 1)

    return portfolio_increment_frame

In [94]:
print(generate_t_increment_portfolio_changes_paths_frame(stocks, mus, sigmas, degrees_of_freedoms, correlations, 10, 10, 3))



       BIDU       TME      VIPS      FTCH        IQ      PARA     DISCA  \
0  0.046122  0.113771  0.149019  0.018228 -0.183132 -0.248708  0.050524   
1 -0.117573  0.586405 -0.209455 -0.076971 -0.032186  0.530620  0.221057   
2 -0.758539 -0.178800 -0.297178 -0.419043 -0.431768 -0.253857 -0.399132   

        GSX       NDX       S&P     Total  
0  0.635282  0.726586  0.965476  2.273169  
1  2.554180  0.037867 -0.133096  3.360847  
2  1.157867 -0.256408 -0.571054 -2.407911  


In [95]:
def generate_correlated_brownians(samples, time_steps, correlated_root, maturity):
    """
    Generates correlated brownians according to a correlation matrix's root
    with maturity T, and time_steps time_steps. Returns samples samples of
    these paths.

    Return is a (samples, time_steps, dimension_of_correlations_matrix) np
    array, so that [i,j,k] is the k'th brownian at the j'th timestep of the i'th
    sample
    """

    n = correlated_root.shape[0] # Dimensions of correlations matrix
    brownian_intervals = np.sqrt(maturity / time_steps) * np.random.standard_normal(size = (time_steps, samples, n))
    correlated_intervals = np.einsum('stj, ij', brownian_intervals, correlated_root).T
    brownian_increments = np.concatenate([np.zeros((samples, 1, n)), correlated_intervals], axis = 1)
    return brownian_increments.cumsum(axis = 1)

In [96]:

n = 2
samples = 4
time_steps = 3
maturity = 1
mu = np.array([1, 2])
vols = np.array([0, 1])


A = mu - 0.5 * vols * vols
print(A)

delta_t = np.linspace(0, maturity, time_steps)
time_space = np.tile(delta_t, reps = (n, 1))
time_space = np.tile(time_space.T, reps = (samples, 1, 1))


multipled = np.einsum('ijk, k -> ijk', time_space, A)
print(multipled[0][2])
print(multipled.shape)


[1.  1.5]
[1.  1.5]
(4, 3, 2)


In [97]:
def generate_correlated_geometric_brownians(samples, time_steps, correlated_root, maturity, mu, vols, initial_values):
    """
    Generates correlated geometric brownians with drifts mu, volatilities vols, and initial values initial_values.

    mu, vols and initial_values should be floats of the same dimension as correlated_root.
    Samples is the number of samples to return
    time_steps is the number of time steps to take
    maturity is the final maturity of the process
    correlated_root is the root (e.g. Cholesky root) of the correlations matrix
    """

    

    n = correlated_root.shape[0]

    # If mu and vols are passed as a pandas data frame, the shape will be (n, 1) rather than (n,)
    # We require shape (n,) for the Einstein sums to work... so we reshape

    if type(mu) == (pd.DataFrame):
        mu = np.reshape(mu.values, (n,))

    if type(vols) == (pd.DataFrame):
        vols = np.reshape(vols.values, (n,))    


    
    brownians = generate_correlated_brownians(samples, time_steps, correlated_root, maturity)

    # Strategy is to generate them from an explicit sampling scheme.
    # i.e. (mu - 0.5 * vol^2)dt  + vol * dW_t
    # Then exponentiate

    delta_t = np.linspace(0, maturity, time_steps + 1)
    time_space = np.tile(delta_t, reps = (n, 1))
    time_space = np.tile(time_space.T, reps = (samples, 1, 1))

    A = mu - 0.5 * vols * vols

    determinstic_increments = np.einsum('ijk, k -> ijk', time_space, A)

    random_increments = np.einsum('ijk, k -> ijk', brownians, vols)

    exponents = determinstic_increments + random_increments

    unitialised_geometric_brownian_motions = np.exp(exponents)

    geometric_brownian_motions = np.einsum('ijk, k -> ijk', unitialised_geometric_brownian_motions, initial_values)

    return geometric_brownian_motions





    

In [98]:
root = np.array([[1,0],[1,0]])
mu = np.array([0.5, 2])
vol = np.array([1, 2])
initial_values = np.array([1, 1])
geometric_brownians = generate_correlated_geometric_brownians(1, 1, root, 1, mu, vol, initial_values)

print("Root Shape")
print(root.shape)
print(mu.shape)
print(vol.shape)


correlations = pd.read_pickle("correlations_2.pkl")
vols = np.reshape(pd.read_pickle("vols.pkl").values, (10,))
means = np.reshape(pd.read_pickle("means.pkl").values, (10,))
cholesky_root = cholesky(correlations)

vols = pd.read_pickle("vols.pkl")
means = pd.read_pickle("means.pkl")

# print(vols.shape)
# print(means.shape)
print(cholesky_root.shape)

all_stock_and_index_values = np.array([266.13, 30.87, 45.25, 62, 28.04, 100.34, 74.65, 83.79, 13086.51, 3940.59])

geometric_brownians = generate_correlated_geometric_brownians(1, 300, cholesky_root, 30/260, means.T, vols.T, all_stock_and_index_values)
print(geometric_brownians)

Root Shape
(2, 2)
(2,)
(2,)
(10, 10)
[[[  266.13          30.87          45.25       ...    83.79
   13086.51        3940.59      ]
  [  269.59761736    30.76605518    46.18187221 ...    85.28998236
   13063.37772536  3947.29089736]
  [  273.13793741    30.88985152    45.42176268 ...    84.35879285
   13076.57381676  3944.54170221]
  ...
  [  346.30451906    38.97600737    72.37500453 ...    95.83227982
   13300.33894975  4054.18402141]
  [  340.15121691    38.66923008    71.70964557 ...    92.86391258
   13387.24051806  4057.71619007]
  [  342.53667512    38.40379252    71.24814997 ...    96.16248829
   13368.63402717  4061.92072725]]]
